# 19 Testing optimized Models after xgb

## Import

In [1]:
import time

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import optuna

import matplotlib as mpl
import matplotlib.pyplot as plt

In [2]:
mpl.style.use("seaborn-v0_8-colorblind")
RANDOM_STATE = 42

## Dataframe

In [3]:
df_raw = fetch_openml(data_id=42742, as_frame=True).frame

In [4]:
feat_cols = [col for col in df_raw.columns if col not in ("target",)]
cat_cols = [col for col in feat_cols if col.endswith("_cat")]
bin_cols = [col for col in feat_cols if col.endswith("_bin")]
num_cols = [col for col in feat_cols if col not in cat_cols and col not in bin_cols]

for col in cat_cols:
    df_raw[col] = df_raw[col].astype("category")

for col in bin_cols:
    if df_raw[col].isna().sum() == 0:
        df_raw[col] = df_raw[col].astype(int).astype("bool")
    else:
        df_raw[col] = df_raw[col].astype("Int8")

for col in num_cols:
    df_raw[col] = df_raw[col].astype("float32")

df_raw["target"] = df_raw["target"].astype(int).astype("bool")

idx_train = np.load("../../../data/processed/train_idx.npy")
idx_val = np.load("../../../data/processed/val_idx.npy")
idx_test = np.load("../../../data/processed/test_idx.npy")

df_train = df_raw.iloc[idx_train]
df_val = df_raw.iloc[idx_val]
df_test = df_raw.iloc[idx_test]

x_train, y_train = df_train[feat_cols], df_train["target"]
x_val, y_val = df_val[feat_cols], df_val["target"]
x_test, y_test = df_test[feat_cols], df_test["target"]
x_full, y_full = df_raw[feat_cols], df_raw["target"]

idx_train_val = np.concatenate([idx_train, idx_val])
df_train_val = df_raw.iloc[idx_train_val]
x_train_val, y_train_val = df_train_val[feat_cols], df_train_val["target"]

pd.Series(
    {
        "train": [len(df_train), len(x_train), len(y_train)],
        "val": [len(df_val), len(x_val), len(y_val)],
        "test": [len(df_test), len(x_test), len(y_test)],
        "full": [len(df_raw), len(x_full), len(y_full)],
        "train_val": [len(df_train_val), len(x_train_val), len(y_train_val)]
    }
)

train        [476168, 476168, 476168]
val             [59522, 59522, 59522]
test            [59522, 59522, 59522]
full         [595212, 595212, 595212]
train_val    [535690, 535690, 535690]
dtype: object

## Hilfsvariablen

In [5]:
results = []
train_times = {}

In [6]:
calc_cols = [c for c in feat_cols if c.startswith("ps_calc_")]

mv_cols = ["ps_car_03_cat", "ps_car_05_cat", "ps_reg_03", "ps_car_14"]

num_cols_no_calc = [c for c in num_cols if c not in calc_cols]
bin_cols_no_calc = [c for c in bin_cols if c not in calc_cols]
feat_cols_no_calc = [c for c in feat_cols if c not in calc_cols]

high_kard_cols = ["ps_car_11_cat"]
low_kard_cols = [c for c in cat_cols if c not in high_kard_cols]

In [7]:
def eval_modell(mod_idx, model, x_train, y_train, x_val, y_val, train_time, best_iter=None):
    """AUC auf Train und Val"""
    auc_train = roc_auc_score(y_train, model.predict_proba(x_train)[:, 1])
    auc_val = roc_auc_score(y_val, model.predict_proba(x_val)[:, 1])
    return {
        "model_idx": mod_idx,
        "auc_train": auc_train,
        "auc_test": auc_val,
        "gini": 2 * auc_val - 1,
        "delta_auc": auc_train - auc_val,
        "best_iter": best_iter,
        "trainingszeit": train_time
    }

In [8]:
def mv_for_cb(df, fill_var="missing"):
    """df copy mit ersetzten nans"""
    outs = df.copy()
    for col in outs.columns:
        if outs[col].dtype.name == "category":
            if fill_var not in outs[col].cat.categories:
                outs[col] = outs[col].cat.add_categories([fill_var])
            outs[col] = outs[col].fillna(fill_var)
    return outs

In [9]:
x_train_cb = mv_for_cb(x_train)
x_val_cb = mv_for_cb(x_val)
x_test_cb = mv_for_cb(x_test)
x_full_cb = mv_for_cb(x_full)
x_train_val_cb = mv_for_cb(x_train_val)

x_train_no_calc_cb = x_train_cb[feat_cols_no_calc]
x_val_no_calc_cb = x_val_cb[feat_cols_no_calc]
x_test_no_calc_cb = x_test_cb[feat_cols_no_calc]
x_full_no_calc_cb = x_full_cb[feat_cols_no_calc]
x_train_val_no_calc_cb = x_train_val_cb[feat_cols_no_calc]

pd.Series(
    {
        "train": [len(x_train_cb), len(x_train_no_calc_cb)],
        "val": [len(x_val_cb), len(x_val_no_calc_cb)],
        "test": [len(x_test_cb), len(x_test_no_calc_cb)],
        "full": [len(x_full_cb), len(x_full_no_calc_cb)],
        "train_val": [len(x_train_val_cb), len(x_train_val_no_calc_cb)]
    }
)

train        [476168, 476168]
val            [59522, 59522]
test           [59522, 59522]
full         [595212, 595212]
train_val    [535690, 535690]
dtype: object

In [10]:
x_train_no_calc = x_train[feat_cols_no_calc]
x_val_no_calc = x_val[feat_cols_no_calc]
x_test_no_calc = x_test[feat_cols_no_calc]

In [11]:
targ_enc = TargetEncoder(cv=5, random_state=RANDOM_STATE)

x_train_te = x_train.copy()
x_val_te = x_val.copy()
x_test_te = x_test.copy()

x_train_te[high_kard_cols] = targ_enc.fit_transform(x_train[high_kard_cols], y_train)
x_val_te[high_kard_cols] = targ_enc.transform(x_val[high_kard_cols])
x_test_te[high_kard_cols] = targ_enc.transform(x_test[high_kard_cols])

x_train_no_calc_te = x_train_te[feat_cols_no_calc]
x_val_no_calc_te = x_val_te[feat_cols_no_calc]
x_test_no_calc_te = x_test_te[feat_cols_no_calc]

x_train_te[high_kard_cols].describe()

c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


,ps_car_11_cat
count,476168.000000
mean,0.036446
std,0.009676
min,0.016329
25%,0.028362
50%,0.034718
75%,0.044178
max,0.078113


In [12]:
def load_cbm(name):
    """cbm modell laden"""
    modell = CatBoostClassifier()
    modell.load_model(f"{name}.cbm")
    return modell

In [13]:
def load_xgb(name):
    """xgb json laden"""
    modell = XGBClassifier()
    modell.load_model(f"{name}.json")
    return modell

## Logreg
Nicht gespeichert, deswegen nochmal training mit:

|Parameter|Wert|
|---|---|
|C|0.01|
|class_weight|balanced|
|fit_intercept|False|
|penalty|None|
|solver|newton-cg|
|tol|0.01|

In [14]:
L_logreg_test_t = make_pipeline(
    make_column_transformer(
            (
                make_pipeline(
                    SimpleImputer(strategy="median", add_indicator=True),
                    StandardScaler()
                ),
                num_cols_no_calc
            ),
            (
                OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"),
                low_kard_cols
            ),
            (
                TargetEncoder(cv=5, shuffle=True, random_state=RANDOM_STATE),
                high_kard_cols
            ),
            (
                "passthrough",
                bin_cols_no_calc
            )
        ),
    LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        C=0.01,
        class_weight="balanced",
        fit_intercept=False,
        penalty=None,
        solver="newton-cg",
        tol=0.01
        )
)

In [15]:
name = "L_logreg_test_t"

start = time.time()
L_logreg_test_t.fit(x_train, y_train)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_logreg_test_t, x_train, y_train, x_test, y_test, train_times[name], best_iter=None)
)

c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\linear_model\_logistic.py:1443: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  wa

In [16]:
L_logreg_test_tv = make_pipeline(
    make_column_transformer(
            (
                make_pipeline(
                    SimpleImputer(strategy="median", add_indicator=True),
                    StandardScaler()
                ),
                num_cols_no_calc
            ),
            (
                OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"),
                low_kard_cols
            ),
            (
                TargetEncoder(cv=5, shuffle=True, random_state=RANDOM_STATE),
                high_kard_cols
            ),
            (
                "passthrough",
                bin_cols_no_calc
            )
        ),
    LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        C=0.01,
        class_weight="balanced",
        fit_intercept=False,
        penalty=None,
        solver="newton-cg",
        tol=0.01
        )
)

In [17]:
name = "L_logreg_test_tv"

start = time.time()
L_logreg_test_tv.fit(x_train_val, y_train_val)
train_times[name] = time.time()- start

results.append(
    eval_modell(name, L_logreg_test_tv, x_train_val, y_train_val, x_test, y_test, train_times[name], best_iter=None)
)

c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\linus\anaconda3\envs\ADA\Lib\site-packages\sklearn\linear_model\_logistic.py:1443: UserWarning: Setting penalty=None will ignore the C and l1_ratio parameters
  wa

## Catboost gespeichert

In [18]:
L_cb_test_01_t = load_cbm("L_cb_opt_01")
L_cb_test_02_t = load_cbm("L_cb_opt_02")
L_cb_test_03_t = load_cbm("L_cb_opt_03")

In [19]:
results.append(
    eval_modell("L_cb_test_01_t", L_cb_test_01_t, x_train_cb, y_train, x_test_cb, y_test, None, best_iter=L_cb_test_01_t.get_best_iteration())
)
results.append(
    eval_modell("L_cb_test_02_t", L_cb_test_02_t, x_train_no_calc_cb, y_train, x_test_no_calc_cb, y_test, None, best_iter=L_cb_test_02_t.get_best_iteration())
)
results.append(
    eval_modell("L_cb_test_03_t", L_cb_test_03_t, x_train_no_calc_cb, y_train, x_test_no_calc_cb, y_test, None, best_iter=L_cb_test_03_t.get_best_iteration())
)

In [20]:
fixed_params_plain = {
    "random_seed": RANDOM_STATE,
    "task_type": "CPU",
    "eval_metric": "AUC",
    "thread_count": -1,
    "allow_writing_files": False,
    "cat_features": cat_cols,
    "use_best_model": True,
    "early_stopping_rounds": 100,
    "loss_function": "Logloss",
    "border_count": 254,
    "verbose": 0,
    "iterations": 5000,
    "boosting_type": "Plain"
}

In [21]:
fixed_params_ordered = {
    "random_seed": RANDOM_STATE,
    "task_type": "CPU",
    "eval_metric": "AUC",
    "thread_count": -1,
    "allow_writing_files": False,
    "cat_features": cat_cols,
    "use_best_model": True,
    "early_stopping_rounds": 100,
    "loss_function": "Logloss",
    "border_count": 254,
    "verbose": 0,
    "iterations": 5000,
    "boosting_type": "Ordered"
}

In [22]:
angepasste_params = [
    "learning_rate", 
    "depth", 
    "l2_leaf_reg", 
    "random_strength", 
    "one_hot_max_size", 
    "leaf_estimation_iterations", 
    "auto_class_weights", 
    "bootstrap_type", 
    "bagging_temperature", 
    "subsample"]

In [23]:
{param : wert for param, wert in L_cb_test_01_t.get_all_params().items() if param in angepasste_params}

{'one_hot_max_size': 105,
 'l2_leaf_reg': 8.593733788,
 'random_strength': 0.01142319106,
 'subsample': 0.9178574681,
 'depth': 7,
 'auto_class_weights': 'Balanced',
 'learning_rate': 0.04761112109,
 'leaf_estimation_iterations': 2,
 'bootstrap_type': 'Bernoulli'}

In [24]:
{param : wert for param, wert in L_cb_test_02_t.get_all_params().items() if param in angepasste_params}

{'one_hot_max_size': 10,
 'l2_leaf_reg': 3.914318323,
 'random_strength': 0.5015455484,
 'subsample': 0.7854715586,
 'depth': 4,
 'auto_class_weights': 'None',
 'learning_rate': 0.1759581715,
 'leaf_estimation_iterations': 2,
 'bootstrap_type': 'Bernoulli'}

In [25]:
{param : wert for param, wert in L_cb_test_03_t.get_all_params().items() if param in angepasste_params}

{'one_hot_max_size': 10,
 'l2_leaf_reg': 21.72735786,
 'random_strength': 1.770355582,
 'depth': 9,
 'bagging_temperature': 0.004430230707,
 'auto_class_weights': 'None',
 'learning_rate': 0.07522925735,
 'leaf_estimation_iterations': 3,
 'bootstrap_type': 'Bayesian'}

## Catboost full saved

In [26]:
L_cb_test_01_t_full = load_cbm("L_cb_opt_01_full")
L_cb_test_02_t_full = load_cbm("L_cb_opt_02_full")
L_cb_test_03_t_full = load_cbm("L_cb_opt_03_full")

In [27]:
results.append(
    eval_modell("L_cb_test_01_t_full", L_cb_test_01_t_full, x_train_cb, y_train, x_test_cb, y_test, None, best_iter=L_cb_test_01_t_full.get_best_iteration())
)
results.append(
    eval_modell("L_cb_test_02_t_full", L_cb_test_02_t_full, x_train_no_calc_cb, y_train, x_test_no_calc_cb, y_test, None, best_iter=L_cb_test_02_t_full.get_best_iteration())
)
results.append(
    eval_modell("L_cb_test_03_t_full", L_cb_test_03_t_full, x_train_no_calc_cb, y_train, x_test_no_calc_cb, y_test, None, best_iter=L_cb_test_03_t_full.get_best_iteration())
)

In [28]:
{param : wert for param, wert in L_cb_test_01_t_full.get_all_params().items() if param in angepasste_params}

{'one_hot_max_size': 10,
 'l2_leaf_reg': 2.430516958,
 'random_strength': 0.001188852475,
 'subsample': 0.5657526255,
 'depth': 5,
 'auto_class_weights': 'Balanced',
 'learning_rate': 0.06338750571,
 'leaf_estimation_iterations': 3,
 'bootstrap_type': 'Bernoulli'}

In [29]:
{param : wert for param, wert in L_cb_test_02_t_full.get_all_params().items() if param in angepasste_params}

{'one_hot_max_size': 18,
 'l2_leaf_reg': 7.076480389,
 'random_strength': 0.8387745023,
 'depth': 8,
 'bagging_temperature': 1.815809965,
 'auto_class_weights': 'None',
 'learning_rate': 0.05445057526,
 'leaf_estimation_iterations': 5,
 'bootstrap_type': 'Bayesian'}

In [30]:
{param : wert for param, wert in L_cb_test_03_t_full.get_all_params().items() if param in angepasste_params}

{'one_hot_max_size': 105,
 'l2_leaf_reg': 18.03974724,
 'random_strength': 0.005187373608,
 'subsample': 0.811945498,
 'depth': 7,
 'auto_class_weights': 'None',
 'learning_rate': 0.1062427759,
 'leaf_estimation_iterations': 1,
 'bootstrap_type': 'Bernoulli'}

## XGB Modelle

In [31]:
L_xgb_opt_01 = load_xgb("L_xgb_opt_01")
L_xgb_opt_02 = load_xgb("L_xgb_opt_02")
L_xgb_opt_03 = load_xgb("L_xgb_opt_03")
L_xgb_opt_04 = load_xgb("L_xgb_opt_04")

In [32]:
results.append(
    eval_modell("L_xgb_opt_01", L_xgb_opt_01, x_train, y_train, x_test, y_test, None, best_iter=L_xgb_opt_01.best_iteration)
)
results.append(
    eval_modell("L_xgb_opt_02", L_xgb_opt_02, x_train_no_calc, y_train, x_test_no_calc, y_test, None, best_iter=L_xgb_opt_02.best_iteration)
)
results.append(
    eval_modell("L_xgb_opt_03", L_xgb_opt_03, x_train_te, y_train, x_test_te, y_test, None, best_iter=L_xgb_opt_03.best_iteration)
)
results.append(
    eval_modell("L_xgb_opt_04", L_xgb_opt_04, x_train_no_calc_te, y_train, x_test_no_calc_te, y_test, None, best_iter=L_xgb_opt_04.best_iteration)
)

In [33]:
study_names = {
    "L_xgb_opt_01": "XGBoost_calc",
    "L_xgb_opt_02": "XGBoost_no_calc",
    "L_xgb_opt_03": "XGBoost_calc_te",
    "L_xgb_opt_04": "XGBoost_no_calc_te"
}

xgb_params = {
    modell: optuna.load_study(study_name=studname, storage="sqlite:///xgboost_opti.db").best_params
    for modell, studname in study_names.items()
}

pd.DataFrame(xgb_params)

,L_xgb_opt_01,L_xgb_opt_02,L_xgb_opt_03,L_xgb_opt_04
learning_rate,0.038768,0.077903,0.018784,0.068409
max_depth,6.000000,7.000000,6.000000,4.000000
min_child_weight,167.926688,126.488318,26.805781,123.572582
gamma,0.033641,0.493110,0.004322,1.226278
reg_lambda,0.462911,2.245830,42.387085,1.262450
reg_alpha,9.390737,0.055327,0.124704,0.389757
max_delta_step,2.000000,7.000000,7.000000,4.000000
subsample,0.940044,0.800055,0.610842,0.871835
colsample_bynode,0.799588,0.896860,0.674854,0.697225
scale_pos_weight,5.000000,1.000000,1.000000,1.000000


## Evaluation

In [34]:
df_results = pd.DataFrame(results).sort_values("auc_test", ascending=False)
df_results

,model_idx,auc_train,auc_test,gini,delta_auc,best_iter,trainingszeit
11,L_xgb_opt_04,0.685711,0.649561,0.299121,0.036151,565.0,NaN
9,L_xgb_opt_02,0.680683,0.649495,0.298989,0.031189,139.0,NaN
8,L_xgb_opt_01,0.736660,0.648169,0.296337,0.088492,729.0,NaN
10,L_xgb_opt_03,0.712847,0.647583,0.295166,0.065264,782.0,NaN
7,L_cb_test_03_t_full,0.685571,0.647561,0.295121,0.038010,789.0,NaN
6,L_cb_test_02_t_full,0.692775,0.647385,0.294769,0.045390,558.0,NaN
3,L_cb_test_02_t,0.664717,0.647273,0.294546,0.017444,518.0,NaN
4,L_cb_test_03_t,0.655782,0.644887,0.289773,0.010896,412.0,NaN
5,L_cb_test_01_t_full,0.696718,0.644159,0.288319,0.052559,623.0,NaN
2,L_cb_test_01_t,0.740788,0.643028,0.286056,0.097760,549.0,NaN


In [35]:
train_times_nb19 = {
    "L_cb_test_01_t": 24.047395,
    "L_cb_test_02_t": 52.476140,
    "L_cb_test_03_t": 192.045585,
    "L_cb_test_01_t_full": 88.705495,
    "L_cb_test_02_t_full": 116.162328,
    "L_cb_test_03_t_full": 124.465556,
    "L_xgb_opt_01": 47.402708,
    "L_xgb_opt_02": 12.787595,
    "L_xgb_opt_03": 62.368462,
    "L_xgb_opt_04": 25.698286,
}

for modell,tt in train_times_nb19.items():
    df_results.loc[df_results["model_idx"] == modell, "trainingszeit"] = tt

df_results

,model_idx,auc_train,auc_test,gini,delta_auc,best_iter,trainingszeit
11,L_xgb_opt_04,0.685711,0.649561,0.299121,0.036151,565.0,25.698286
9,L_xgb_opt_02,0.680683,0.649495,0.298989,0.031189,139.0,12.787595
8,L_xgb_opt_01,0.736660,0.648169,0.296337,0.088492,729.0,47.402708
10,L_xgb_opt_03,0.712847,0.647583,0.295166,0.065264,782.0,62.368462
7,L_cb_test_03_t_full,0.685571,0.647561,0.295121,0.038010,789.0,124.465556
6,L_cb_test_02_t_full,0.692775,0.647385,0.294769,0.045390,558.0,116.162328
3,L_cb_test_02_t,0.664717,0.647273,0.294546,0.017444,518.0,52.476140
4,L_cb_test_03_t,0.655782,0.644887,0.289773,0.010896,412.0,192.045585
5,L_cb_test_01_t_full,0.696718,0.644159,0.288319,0.052559,623.0,88.705495
2,L_cb_test_01_t,0.740788,0.643028,0.286056,0.097760,549.0,24.047395


## Notizen
- bei xgb muss ich die params aus optuna holen get params funktioniert hier nicht richtig
- Target Encoding scheint dem xgboost nichts zu bringen (extrem wenig gini diff 02 zu 04)
- Calc drop zeigt sehr positive effekte --> Rechenzeit verbessert und delta auc viel kleiner --> weniger overfitting auf train

## Beste Modelle bis jetzt
Ich würde bis jetzt die Modelle L_xgb_opt_02 , L_cb_test_02_t und L_cb_test_03_t_full bzw L_cb_test_02_t_full als beste Modelle darstellen. (Ein sehr gutes Logreg fehlt noch für eventuell leichtes gewicht in Ensemble) <br>

L_xgb_opt_02:
```python
#data no calc
L_xgb_opt_02 = XGBClassifier(
    random_state=42,
    verbosity=0,
    device="cpu",
    n_jobs=-1,
    tree_method="hist",
    objective="binary:logistic",
    eval_metric="auc",
    enable_categorical=True,
    n_estimators=3000,
    early_stopping_rounds=100,
    learning_rate=0.077903,
    max_depth=7,
    min_child_weight=126.488318,
    gamma=0.493110,
    reg_lambda=2.245830,
    reg_alpha=0.055327,
    max_delta_step=7,
    subsample=0.800055,
    colsample_bynode=0.896860,
    scale_pos_weight=1,
    max_cat_to_onehot=105
)

L_xgb_opt_02.fit(
    x_train_no_calc, y_train,
    eval_set=[(x_val_no_calc, y_val)],
    verbose=250
)
```

L_cb_test_02_t
```python
# data ohne calc, missing kategorie in cat features statt nan
L_cb_opt_02 = CatBoostClassifier(
    random_seed=42,
    task_type="CPU",
    thread_count=-1,
    eval_metric="AUC",
    loss_function="Logloss",
    allow_writing_files=False,
    cat_features=cat_cols,
    use_best_model=True,
    early_stopping_rounds=100,
    border_count=254,
    verbose=0,
    iterations=5000,
    boosting_type="Plain",
    learning_rate=0.1759581715,
    depth=4,
    l2_leaf_reg=3.914318323,
    random_strength=0.5015455484,
    one_hot_max_size=10,
    leaf_estimation_iterations=2,
    auto_class_weights=None,
    bootstrap_type="Bernoulli",
    subsample=0.7854715586
)

L_cb_opt_02.fit(
    x_train_no_calc_cb, y_train,
    eval_set=(x_val_no_calc_cb, y_val),
    verbose=250
)
```

L_cb_test_03_t_full
```python
# data ohne calc, missing kategorie in cat features statt nan
L_cb_opt_03_full = CatBoostClassifier(
    random_seed=42,
    task_type="CPU",
    thread_count=-1,
    eval_metric="AUC",
    loss_function="Logloss",
    allow_writing_files=False,
    cat_features=cat_cols,
    use_best_model=True,
    early_stopping_rounds=100,
    border_count=254,
    verbose=0,
    iterations=5000,
    boosting_type="Ordered",
    learning_rate=0.1062427759,
    depth=7,
    l2_leaf_reg=18.03974724,
    random_strength=0.005187373608,
    one_hot_max_size=105,
    leaf_estimation_iterations=1,
    auto_class_weights=None,
    bootstrap_type="Bernoulli",
    subsample=0.811945498
)

L_cb_opt_03_full.fit(
    x_train_no_calc_cb, y_train,
    eval_set=(x_val_no_calc_cb, y_val),
    verbose=250
)
```

L_cb_test_02_t_full
```python
# data ohne calc, missing kategorie in cat features statt nan
L_cb_opt_02_full = CatBoostClassifier(
    random_seed=42,
    task_type="CPU",
    thread_count=-1,
    eval_metric="AUC",
    loss_function="Logloss",
    allow_writing_files=False,
    cat_features=cat_cols,
    use_best_model=True,
    early_stopping_rounds=100,
    border_count=254,
    verbose=0,
    iterations=5000,
    boosting_type="Plain",
    learning_rate=0.05445057526,
    depth=8,
    l2_leaf_reg=7.076480389,
    random_strength=0.8387745023,
    one_hot_max_size=18,
    leaf_estimation_iterations=5,
    auto_class_weights=None,
    bootstrap_type="Bayesian",
    bagging_temperature=1.815809965
)

L_cb_opt_02_full.fit(
    x_train_no_calc_cb, y_train,
    eval_set=(x_val_no_calc_cb, y_val),
    verbose=250
)
```